In [36]:
from astroquery.gaia import Gaia
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from astroquery.vizier import Vizier
import astropy.units as u
from astropy.coordinates import Angle

from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic
from astropy.coordinates import Latitude, Longitude
from scipy.optimize import minimize
from astropy.coordinates import FK5

In [ ]:
# zvezde spektralnog tipa O (Teff >= 30,000 K)
# uzimamo samo 2000

o_query = """
SELECT TOP 3000
    g.source_id, 
    g.ra, 
    g.dec, 
    g.parallax, 
    g.parallax_over_error, 
    g.pmra, 
    g.pmdec, 
    g.phot_g_mean_mag, 
    g.bp_rp,
    a.teff_gspphot
FROM gaiadr3.gaia_source AS g
JOIN gaiadr3.astrophysical_parameters AS a ON g.source_id = a.source_id
WHERE a.teff_gspphot >= 30000
  AND g.parallax > 0
  AND g.parallax_over_error > 5
"""

job_o = Gaia.launch_job_async(o_query)
df_o = job_o.get_results().to_pandas()
df_o.to_csv("o_stars.csv", index=False)

In [ ]:
# učitavam podatke za O: ra, dec i parallax
df_o = pd.read_csv("o_stars.csv")
o_data = df_o[["ra", "dec", "parallax"]]

In [ ]:
# učitavamo celi katalog - Vizier ima svoj row limit
Vizier.ROW_LIMIT = -1

# globularna jata
globular_result = Vizier.get_catalogs("VII/202")
df_globular = globular_result[0].to_pandas()

# zadržavam samo podatke koji su mi potrebni
df_globular_clean = df_globular[["ID", "RAJ2000", "DEJ2000"]]
df_globular_clean.to_csv("globular_clusters_harris.csv", index=False)

# otvorena jata
open_result = Vizier.get_catalogs("B/ocl")
df_open = open_result[0].to_pandas()

# zadržavam samo podatke koji su mi potrebni
df_open_clean = df_open[["RAJ2000", "DEJ2000", "Dist"]]
df_open_clean.to_csv("open_clusters_dias.csv", index=False)

# 1. zadatak

In [ ]:
plt.figure()

plt.scatter(Angle(df_open_clean["RAJ2000"].astype(str), unit=u.hourangle).degree,
            Angle(df_open_clean["DEJ2000"].astype(str), unit=u.deg).degree, label='Otvorena zvezdana jata', alpha=0.5, color='#3D3B8E')

plt.scatter(o_data['ra'], o_data['dec'], label='O zvezde', color='#E072A4', alpha=0.5)

plt.legend()
plt.xlabel('RA')
plt.ylabel('DEC')

plt.show()

U prvom zadatku urađen je scatter podataka za O zvezde i otvorena zvezdana jata.

Kako su podaci za otvorena zvezdana jata dati u formatima h:m:s za rektascenziju i d:m:s za deklinaciju, iskorišćena je  funkcija .degree.

unit=u.hourangle za rektascenziju u časovnim uglovima i unit=u.deg za deklinaciju u stepenima, lučnim minutima i lučnim sekundama vraćaju decimalne stepene.

# 2. zadatak

In [ ]:
def udaljenost(paralaksa):
    return 1/(paralaksa/1000)
    
o_data["distance_pc"] = udaljenost(o_data["parallax"])

print(o_data["distance_pc"])

o_data.to_csv("udaljenost_do_o_zvezda.csv", index=False)

U drugom zadatku izračunata je udaljenost do O zvezda pomoću formule <span style="color: #E70E02;">$r = \frac{1}{\pi}$</span>.

<span style="color: #99DDC8;">parallax : Parallax (double, Angle[mas] )</span>

Paralaksa iz kataloga je podeljena sa 1000 da bi se iz mas prešlo u as

Rezultati se poklapaju sa podacima sa stranice https://en.wikipedia.org/wiki/List_of_O-type_stars

# 3. zadatak

In [ ]:
def veliki_krug(ra, alphaw, i):
    ra = np.deg2rad(ra)
    alphaw = np.deg2rad(alphaw)
    i = np.deg2rad(i)

    dec=np.arctan(np.sin(ra-alphaw)*np.tan(i))

    return np.rad2deg(dec)

In [ ]:
o_stars = pd.DataFrame({"RA": o_data["ra"],"DE": o_data["dec"]})
open_data = pd.DataFrame({"RA": Angle(df_open_clean["RAJ2000"].astype(str), unit=u.hourangle).degree,
                          'DE':Angle(df_open_clean["DEJ2000"].astype(str), unit=u.deg).degree})

data = pd.concat([o_stars,open_data], ignore_index=True)

ra = data["RA"].to_numpy(dtype=float)
dec = data["DE"].to_numpy(dtype=float)

In [ ]:
def ls(alphaw,i):
    dec_model=veliki_krug(ra, alphaw, i)
    razlika=dec-dec_model
    S=np.sum(razlika**2)
    return S

In [ ]:
result=minimize(
    lambda parameters: ls(parameters[0], parameters[1]),
    x0=[60,60], bounds=[(-180, 180), (0.1, 89.9)] #0.1 i 89.9 zbog definisanosti tangensa
)

alphaw_fit, i_fit = result.x

if alphaw_fit<0:
    alphaw_fit=alphaw_fit+360

alphaw_hms=Angle(alphaw_fit%360,unit=u.deg).to_string(unit=u.hourangle)

print("Najbolje alphaw:", alphaw_hms)
print("Najbolje i:", i_fit)
print("Minimalno S:", result.fun)

Dobijene vrednosti za i i alphaw su približno jednake usvojenim vrednostima za epohu J2000.

In [ ]:
def alpha_png(alphaw):
    alphaw=alphaw/15
    return alphaw-6

def delta_png(i):
    return 90-i

alphapng=alpha_png(alphaw_fit)
deltapng=delta_png(i_fit)

print('Položaj galaktičkog pola- rektascenzija (h):',alphapng, ', deklinacija (deg):',deltapng)

Dobijene vrednosti su približno jednake vrednostima ekvatorskih koordinata severnog galaktičkog pola - ra: 12 51 26.28, dec: 27 07 42

In [ ]:
ra_model = np.linspace(0, 360, 1000)
dec_model = veliki_krug(ra_model, alphaw_fit, i_fit)

plt.figure()
plt.scatter(ra, dec, alpha=0.5, label="Posmatrani podaci", color='#373F51')
plt.plot(ra_model, dec_model, label="Fitovani veliki krug", color='#E43F6F')

plt.xlabel("Rektascenzija")
plt.ylabel("Deklinacija")
plt.xlim(0, 360)
plt.ylim(-90, 90)
plt.legend()
plt.show()

# 4. zadatak

In [ ]:
coordinates=SkyCoord(df_globular_clean['RAJ2000'].astype(str),df_globular_clean['DEJ2000'].astype(str),
                    unit=(u.hourangle, u.deg),frame='icrs')

galactic=coordinates.galactic

df_globular_clean["l"] = galactic.l.deg
df_globular_clean["b"] = galactic.b.deg

print(df_globular_clean[["l", "b"]])

df_globular_clean.to_csv("globularna_jata_galakticke_koordinate.csv", index=False)

U četvrtom zadatku su ekvatorske koordinate prevedene u galaktičke pomoću .galactic funkcije SkyCoord-a.

# 5. zadatak

In [ ]:
bj=[]
bjc=[]
for j in range(1,12): #jer da počinje od nule bilo bi za j=0 arcsin(-1.2), što nije interval za arcsin
                      #12 jer za 10 oblasti imam 11 granica
    bj.append(np.asin(0.2*(j-6)))
for j in range(1,11):
    bjc.append(np.asin(0.2*(j-6)+0.1))

print(bj)
print()
print(np.degrees(bjc))

In [ ]:
df=pd.read_csv('globularna_jata_galakticke_koordinate.csv')
b=df['b'].to_numpy()

bj=np.degrees(bj)


n=[]
for i in range(10):
    n.append(np.sum(
        (b>bj[i]) & (b<bj[i+1])
    ))

print(n)
print()
print(np.sum(n), b.size)

In [ ]:
#napravila sam histogram pomoću pravougaonika jer nisam znala da podesim širine kako treba
plt.figure()

for i in range(10):
    leva=bj[i]
    desna=bj[i+1]
    visina=n[i]
    
    plt.plot([leva, desna], [0, 0], color='#9C528B')
    plt.plot([leva, leva], [0, visina], color='#9C528B')
    plt.plot([leva, desna], [visina, visina], color='#9C528B')
    plt.plot([desna, desna], [0, visina], color='#9C528B')

plt.xlabel('Galaktička širina b [°]')
plt.ylabel('Broj globularnih jata')
plt.xlim(-90, 90)
plt.show()

In [ ]:
bjc=np.degrees(bjc)

bmax=(np.sum(n*bjc))/(np.sum(n))
print('bmax=', bmax)

### Podela od 20 zona

In [ ]:
#prekopiran gornji kod, ali zamenjeni brojevi stubića
#h=0.1 i polovina visine je 0.05
#srednji indeks je j=11 (za koji je sinb=0)

bj=[]
bjc=[]
for j in range(1,22):
    bj.append(np.asin(0.1*(j-11)))
for j in range(1,21):
    bjc.append(np.asin(0.1*(j-11)+0.05))

print(bj)
print()
print(np.degrees(bjc))
print()

df=pd.read_csv('globularna_jata_galakticke_koordinate.csv')
b=df['b'].to_numpy()

bj=np.degrees(bj)


n=[]
for i in range(20):
    n.append(np.sum(
        (b>bj[i]) & (b<bj[i+1])
    ))

print(n)
print()
print(np.sum(n), b.size)

plt.figure()

for i in range(20):
    leva=bj[i]
    desna=bj[i+1]
    visina=n[i]
    
    plt.plot([leva, desna], [0, 0], color='#9C528B')
    plt.plot([leva, leva], [0, visina], color='#9C528B')
    plt.plot([leva, desna], [visina, visina], color='#9C528B')
    plt.plot([desna, desna], [0, visina], color='#9C528B')

plt.xlabel('Galaktička širina b [°]')
plt.ylabel('Broj globularnih jata')
plt.xlim(-90, 90)
plt.show()

bjc=np.degrees(bjc)

bmax20=(np.sum(n*bjc))/(np.sum(n))
print('bmax20=', bmax20)

Očekivali bismo da je $b_{\max} = 0^\circ$ , budući da je to latituda koja odgovara disku Galaksije, no vidimo da postoje odstupanja: $b_{\max10} = 1.29^\circ$ i $b_{\max20} = 1.53^\circ$

# 6. zadatak

In [ ]:
lj=[]
ljc=[]
for j in range(0,11):
    lj.append(-180+j*36) #360/10=36
for j in range(0,10):
    ljc.append(-180+j*36+18) #36/2=18

print(lj)
print()
print('ljc:',ljc)
print()


df=pd.read_csv('globularna_jata_galakticke_koordinate.csv')
l=df['l'].to_numpy()

l=(l+180)%360-180 #jer je u katalogu od 0 do 360, a interval -180, 180 mi treba za računanje lmax

n=[]
for i in range(10):
    n.append(np.sum(
        (l>=lj[i]) & (l<lj[i+1])
    ))

print(n)
print()
print(np.sum(n), l.size)
print()


plt.figure()

for i in range(10):
    leva=lj[i]
    desna=lj[i+1]
    visina=n[i]
    
    plt.plot([leva, desna], [0, 0], color='#9C528B')
    plt.plot([leva, leva], [0, visina], color='#9C528B')
    plt.plot([leva, desna], [visina, visina], color='#9C528B')
    plt.plot([desna, desna], [0, visina], color='#9C528B')

plt.xlabel('Galaktička dužina l [°]')
plt.ylabel('Broj globularnih jata')
plt.xlim(-180, 180)
plt.show()

lmax=(np.sum(np.array(n)*np.array(ljc)))/(np.sum(n))
print('lmax=', lmax)

### 20 stubova

In [ ]:
lj=[]
ljc=[]
for j in range(0,21):
    lj.append(-180+j*18) #360/20=18
for j in range(0,20):
    ljc.append(-180+j*18+9) #18/2=9

print(lj)
print()
print('ljc:',ljc)
print()


df=pd.read_csv('globularna_jata_galakticke_koordinate.csv')
l=df['l'].to_numpy()

l=(l+180)%360-180

n=[]
for i in range(20):
    n.append(np.sum(
        (l>=lj[i]) & (l<lj[i+1])
    ))

print(n)
print()
print(np.sum(n), l.size)
print()


plt.figure()

for i in range(20):
    leva=lj[i]
    desna=lj[i+1]
    visina=n[i]
    
    plt.plot([leva, desna], [0, 0], color='#9C528B')
    plt.plot([leva, leva], [0, visina], color='#9C528B')
    plt.plot([leva, desna], [visina, visina], color='#9C528B')
    plt.plot([desna, desna], [0, visina], color='#9C528B')

plt.xlabel('Galaktička dužina l [°]')
plt.ylabel('Broj globularnih jata')
plt.xlim(-180, 180)
plt.show()


lmax20=(np.sum(np.array(n)*np.array(ljc)))/(np.sum(n))
print('lmax20=', lmax20)

Ovde nisam mogla da koristim arcsin zbog njegove oblasti definisanosti, no su zone jednake širine po uglu.

Ostatak zadatka je sličan prethodnom.

# 7. zadatak

Očekivane vrednosti su $b_{\max} = 0^\circ$ i $l_{\max} = 0^\circ$

Ukoliko Galaksiju podelimo na 10 oblasti, dobijamo $b_{\max10} = 1.29^\circ$ i $l_{\max10} = -3.55^\circ$ , a ukoliko je podelimo na 20, dobijamo $b20_{\max} = 1.53^\circ$ i $l_{\max20} = -4.35^\circ$ , što je približno jednako očekivanoj vrednosti. Odstupanje može biti posledica veličine uzorka i načina određivanja maksimuma.

In [35]:
centar=SkyCoord(l=0*u.deg, b=0*u.deg, frame='galactic')

ekv=centar.transform_to(FK5(equinox='J2000'))

racG=ekv.ra.to(u.hourangle)
deccG=ekv.dec.to(u.deg)

print("RA =", racG)
print("DEC =", deccG)

RA = 17h45m37.19909617s
DEC = -28d56m10.22065221s


Sagittarius_A*: https://en.wikipedia.org/wiki/Sagittarius_A*

Epoch J2000  
Right ascension	17h 45m 40.0409s  
Declination	−29$\circ$ 0′ 28.118″

Izračunate koordinate centra Galaksije su probližno jednake stvarnim koordinatama centra Galaksije (supermasivne crne rupe Sagittarius A*
).